# Task 1 + Task 2 — XGBoost pipeline
- **Task 1 Anti-Cheat** (binary, **F2 macro**): XGB 5-fold → หา threshold ที่ให้ F2 macro สูงสุดบน OOF
- **Task 2 Segment** (4 คลาส, **F1 macro**): XGB 5-fold + class weight → ปรับน้ำหนักต่อคลาสบน OOF ให้ F1 macro สูงสุด
- Baseline HGB เดิม: T1 F2 macro 0.778 · T2 F1 macro 0.742
**ไม่ submit** — เขียนไฟล์ลง `/kaggle/working/preds/` · แนบ Input: competition `player-intelligence-system-cpe-342-ml`

In [ ]:
import os, time, warnings, numpy as np, pandas as pd, xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import fbeta_score, f1_score, classification_report, confusion_matrix
warnings.filterwarnings("ignore")
SEED, NF = 42, 5
N_EST, ES = 4000, 150
DEVICE = "cuda" if os.system("nvidia-smi > /dev/null 2>&1") == 0 else "cpu"
print("xgboost", xgb.__version__, "| device", DEVICE)
ROOT = "/kaggle/input"
def walk_find(name, must=""):
    for d, _, fs in os.walk(ROOT, followlinks=True):
        if name in fs and must in d: return os.path.join(d, name)
SAMPLE = walk_find("sample_submission.csv")
print("sample:", SAMPLE)
os.makedirs("/kaggle/working/preds", exist_ok=True)

def run_cv(X, y, Xt, params, sample_weight=None, n_class=2):
    folds = list(StratifiedKFold(NF, shuffle=True, random_state=SEED).split(X, y))
    oof = np.zeros((len(y), n_class)); te = np.zeros((len(Xt), n_class)); imp = pd.Series(0.0, index=X.columns)
    for k, (a, b) in enumerate(folds):
        t0 = time.time()
        m = xgb.XGBClassifier(**params).fit(X.iloc[a], y[a], sample_weight=None if sample_weight is None else sample_weight[a],
                                            eval_set=[(X.iloc[b], y[b])], verbose=False)
        oof[b] = m.predict_proba(X.iloc[b]); te += m.predict_proba(Xt) / NF
        imp += pd.Series(m.feature_importances_, index=X.columns) / NF
        print(f"  fold {k}: best_iter {m.best_iteration} | {time.time()-t0:.0f}s")
    return oof, te, imp

def write_pred(task, pred):
    out = pd.DataFrame({"row": np.arange(len(pred)), "pred": pred})
    if SAMPLE:
        ids = pd.read_csv(SAMPLE)["id"]; assert len(ids) == len(pred); out.insert(0, "id", ids.values)   # จับคู่ตามลำดับแถว
    out.to_csv(f"/kaggle/working/preds/{task}_xgb.csv", index=False); print("wrote", f"preds/{task}_xgb.csv")

## Task 1 — Anti-Cheat (F2 macro)

In [ ]:
tr = pd.read_csv(walk_find("train.csv", "01_binary")); te = pd.read_csv(walk_find("test.csv", "01_binary"))
tr = tr[tr["is_cheater"].notna()].reset_index(drop=True)          # target ว่าง ~12.7% → ตัดทิ้ง
y1 = tr["is_cheater"].astype(int).values
def fe1(d):
    d = d.copy(); e = 1e-3
    d["kd_x_hs"]          = d["kill_death_ratio"] * d["headshot_percentage"]
    d["acc_per_react"]    = d["accuracy_score"] / (d["reaction_time_ms"] + e)
    d["kd_per_level"]     = d["kill_death_ratio"] / (d["level"] + 1)
    d["reports_per_day"]  = d["reports_received"] / (d["account_age_days"] + 1)
    d["level_per_day"]    = d["level"] / (d["account_age_days"] + 1)
    d["aim_combo"]        = d["aiming_smoothness"] * d["crosshair_placement"]
    d["skill_vs_sense"]   = d["accuracy_score"] - d["game_sense_score"]          # เล็งเก่งแต่อ่านเกมไม่เก่ง = น่าสงสัย
    d["devices_per_day"]  = d["device_changes_count"] / (d["account_age_days"] + 1)
    return d
F1 = [c for c in fe1(te).columns if c != "player_id"]
X1, X1t = fe1(tr)[F1], fe1(te)[F1]
print(X1.shape, "| cheater rate", y1.mean().round(3))
P1 = dict(objective="binary:logistic", eval_metric="logloss", n_estimators=N_EST, learning_rate=0.03, max_depth=6,
          min_child_weight=3, subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0, tree_method="hist", device=DEVICE,
          early_stopping_rounds=ES, random_state=SEED)
oof1, te1, imp1 = run_cv(X1, y1, X1t, P1)
f2m = lambda yt, yp: fbeta_score(yt, yp, beta=2, average="macro")
ths = np.arange(0.05, 0.95, 0.01); sc = [f2m(y1, (oof1[:, 1] >= t).astype(int)) for t in ths]
T1_THR = float(ths[int(np.argmax(sc))])
print(f"\nF2 macro @0.5 = {f2m(y1, (oof1[:,1]>=.5).astype(int)):.4f} | best thr {T1_THR:.2f} → F2 macro {max(sc):.4f}"
      f" | F2 (class 1 only) {fbeta_score(y1, (oof1[:,1]>=T1_THR).astype(int), beta=2):.4f}   (baseline HGB 0.778)")
print(imp1.sort_values(ascending=False).head(12).round(4).to_string())
write_pred("t1", (te1[:, 1] >= T1_THR).astype(int))
np.save("/kaggle/working/preds/t1_oof.npy", oof1); np.save("/kaggle/working/preds/t1_test_proba.npy", te1)

## Task 2 — Player Segment (F1 macro)

In [ ]:
tr = pd.read_csv(walk_find("train.csv", "02_multiclass")); te = pd.read_csv(walk_find("test.csv", "02_multiclass"))
y2 = tr["segment"].astype(int).values
DROP_RANDOM = True       # random_metric_1..3 ชื่อบอกว่าเป็น noise → ลองทั้ง True/False
CATS = [c for c in te.columns if not pd.api.types.is_numeric_dtype(te[c]) and c != "player_id"]
def fe2(d):
    d = d.copy(); e = 1.0
    d["spend_per_hour"]    = d["total_spending_thb"] / (d["total_playtime_hours"] + e)
    d["spend_per_day"]     = d["total_spending_thb"] / (d["account_age_days"] + e)
    d["social_sum"]        = d["friend_count"] + d["friend_invites_sent"] + d["gifts_sent_received"]
    d["social_per_hour"]   = d["social_sum"] / (d["total_playtime_hours"] + e)
    d["comp_score"]        = d["ranked_participation_rate"] * d["win_rate_ranked"]
    d["cosmetic_vs_perf"]  = d["item_type_preference_cosmetic"] - d["item_type_preference_performance"]
    d["session_x_freq"]    = d["avg_session_duration"] * d["play_frequency"]
    for c in CATS: d[c] = d[c].astype("category")
    return d
F2 = [c for c in fe2(te).columns if c != "player_id" and not (DROP_RANDOM and c.startswith("random_metric"))]
all2 = pd.concat([fe2(tr)[F2], fe2(te)[F2]], keys=["tr", "te"])          # category เดียวกันทั้ง train/test
for c in CATS: all2[c] = all2[c].astype(str).astype("category")
X2, X2t = all2.loc["tr"], all2.loc["te"]
cnt = np.bincount(y2); sw = (len(y2) / (len(cnt) * cnt))[y2]            # class weight (macro F1)
print(X2.shape, "| class dist", (cnt / cnt.sum()).round(3))
P2 = dict(objective="multi:softprob", num_class=4, eval_metric="mlogloss", n_estimators=N_EST, learning_rate=0.03,
          max_depth=6, min_child_weight=3, subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0, tree_method="hist",
          device=DEVICE, enable_categorical=True, max_cat_to_onehot=8, early_stopping_rounds=ES, random_state=SEED)
oof2, te2, imp2 = run_cv(X2, y2, X2t, P2, sample_weight=sw, n_class=4)
base = f1_score(y2, oof2.argmax(1), average="macro")
# ปรับน้ำหนักต่อคลาส (coordinate search) ให้ F1 macro สูงสุดบน OOF
W = np.ones(4)
for _ in range(3):
    for c in range(4):
        best_w, best_s = W[c], f1_score(y2, (oof2 * W).argmax(1), average="macro")
        for w in np.arange(0.6, 1.61, 0.05):
            W2 = W.copy(); W2[c] = w; s = f1_score(y2, (oof2 * W2).argmax(1), average="macro")
            if s > best_s: best_w, best_s = w, s
        W[c] = best_w
tuned = f1_score(y2, (oof2 * W).argmax(1), average="macro")
print(f"\nF1 macro (argmax) {base:.4f} → class weights {W.round(2)} → {tuned:.4f}   (baseline HGB 0.742)")
print(classification_report(y2, (oof2 * W).argmax(1), digits=4))
print(pd.DataFrame(confusion_matrix(y2, (oof2 * W).argmax(1))))
print(imp2.sort_values(ascending=False).head(12).round(4).to_string())
write_pred("t2", (te2 * W).argmax(1))
np.save("/kaggle/working/preds/t2_oof.npy", oof2); np.save("/kaggle/working/preds/t2_test_proba.npy", te2)